In [ ]:
tenant_id: str = "acme"
batch_date: str = "2026-06-08"


In [ ]:
%pip install azure-ai-documentintelligence azure-keyvault-secrets azure-identity --quiet


In [ ]:
import os
import sys

# fabric git sync places workspace files here - project root must be on path for extractor/quality imports
sys.path.insert(0, "/home/trusted-service-user/work/meridian")


In [ ]:
from dataclasses import asdict

from pyspark.sql.functions import current_timestamp, lit

from extractor.layout import extract_sections
from extractor.splitter import SectionSplitter
from quality.suite import run_suite


In [ ]:
from azure.identity import ManagedIdentityCredential
from azure.keyvault.secrets import SecretClient

_kv = SecretClient(
    vault_url="https://meridian-kv-rk1.vault.azure.net/",
    credential=ManagedIdentityCredential(),
)
DOCINTEL_ENDPOINT: str = _kv.get_secret("docintel-endpoint").value
DOCINTEL_KEY: str = _kv.get_secret("docintel-key").value

BRONZE_PATH: str = f"/lakehouse/default/Files/bronze/{tenant_id}/{batch_date}"
SILVER_CHUNKS_TABLE: str = "silver_doc_chunks"
SILVER_REJECTED_TABLE: str = "silver_rejected"


In [ ]:
SUPPORTED_EXTENSIONS = (".pdf", ".docx", ".xlsx", ".pptx")

raw_files = [
    os.path.join(BRONZE_PATH, f)
    for f in os.listdir(BRONZE_PATH)
    if f.lower().endswith(SUPPORTED_EXTENSIONS)
]

if not raw_files:
    raise RuntimeError(f"no documents found in {BRONZE_PATH}")

print(f"{len(raw_files)} documents queued  tenant={tenant_id}  date={batch_date}")


In [ ]:
splitter = SectionSplitter()
all_chunks = []

# serial - document intelligence S0 tier enforces per-minute request limits
for file_path in raw_files:
    doc_id = os.path.splitext(os.path.basename(file_path))[0]
    sections = extract_sections(
        file_path=file_path,
        endpoint=DOCINTEL_ENDPOINT,
        key=DOCINTEL_KEY,
        tenant_id=tenant_id,
        doc_id=doc_id,
    )
    chunks = splitter.split(sections, tenant_id=tenant_id, doc_id=doc_id)
    all_chunks.extend(chunks)

print(f"{len(all_chunks)} chunks extracted from {len(raw_files)} documents")


In [ ]:
batch_df = spark.createDataFrame([asdict(c) for c in all_chunks])


In [ ]:
pass_df, rejected_df = run_suite(batch_df)

pass_count = pass_df.count()
rejected_count = rejected_df.count()
total = pass_count + rejected_count

print(f"dq result  passed={pass_count}  rejected={rejected_count}  pass_rate={pass_count / max(total, 1):.1%}")


In [ ]:
(
    pass_df
    .withColumn("ingested_at", current_timestamp())
    .withColumn("batch_date", lit(batch_date))
    .write.format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(SILVER_CHUNKS_TABLE)
)

(
    rejected_df
    .withColumn("ingested_at", current_timestamp())
    .withColumn("batch_date", lit(batch_date))
    .write.format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(SILVER_REJECTED_TABLE)
)


In [ ]:
print("--- batch summary ---")
print(f"tenant:           {tenant_id}")
print(f"batch_date:       {batch_date}")
print(f"documents:        {len(raw_files)}")
print(f"chunks extracted: {len(all_chunks)}")
print(f"silver written:   {pass_count}")
print(f"rejected:         {rejected_count}")
print(f"pass rate:        {pass_count / max(total, 1):.1%}")
